# 🔍 Bloque 0 — Auditoría de Calidad de Datos con SQL

Usamos **DuckDB** 🦆 — lee CSVs directamente con SQL estándar.
Las mismas queries funcionan en **BigQuery** cambiando solo el nombre de las tablas.

---

## ⚙️ Setup — Conectar DuckDB y registrar los CSV como tablas

In [ ]:
import duckdb
import pandas as pd

# Conexion en memoria (no necesita servidor)
con = duckdb.connect()

PATH = r"C:\Users\d0c00v5\Downloads\Datasets_extracted"

# Registrar cada CSV como una tabla virtual
con.execute(f"CREATE VIEW stores       AS SELECT * FROM read_csv_auto('{PATH}/stores.csv')")
con.execute(f"CREATE VIEW products     AS SELECT * FROM read_csv_auto('{PATH}/products.csv')")
con.execute(f"CREATE VIEW vendors      AS SELECT * FROM read_csv_auto('{PATH}/vendors.csv')")
con.execute(f"CREATE VIEW promotions   AS SELECT * FROM read_csv_auto('{PATH}/store_promotions.csv')")
con.execute(f"CREATE VIEW transactions AS SELECT * FROM read_csv_auto('{PATH}/transactions.csv')")
con.execute(f"CREATE VIEW tx_items     AS SELECT * FROM read_csv_auto('{PATH}/transaction_items.csv')")

# Funcion helper para correr SQL y ver resultado bonito
def sql(query):
    return con.execute(query).df()

print('DuckDB listo! Tablas registradas:')sql("SHOW TABLES")

In [ ]:
# Resumen rapido de cada tabla
sql("""
SELECT 'transactions' AS tabla, COUNT(*) AS filas FROM transactions
UNION ALL
SELECT 'tx_items',     COUNT(*) FROM tx_items
UNION ALL
SELECT 'stores',       COUNT(*) FROM stores
UNION ALL
SELECT 'products',     COUNT(*) FROM products
UNION ALL
SELECT 'vendors',      COUNT(*) FROM vendors
UNION ALL
SELECT 'promotions',   COUNT(*) FROM promotions
""")

#### ---
## 1️⃣ COMPLETITUD — ¿Faltan datos?
> **Dimension:** Completitud
> **Pregunta:** ¿Qué % de transacciones no tiene `customer_id` o tiene `loyalty_card = FALSE`?

In [ ]:
sql("""
SELECT
    COUNT(*)                                         AS total_transacciones,

    -- customer_id nulo
    COUNT(*) FILTER (WHERE customer_id IS NULL)      AS sin_customer_id,
    ROUND(
        COUNT(*) FILTER (WHERE customer_id IS NULL)
        * 100.0 / COUNT(*), 1
    )                                                AS pct_sin_customer,

    -- loyalty card
    COUNT(*) FILTER (WHERE loyalty_card = false)     AS loyalty_false,
    COUNT(*) FILTER (WHERE loyalty_card = true)      AS loyalty_true,
    ROUND(
        COUNT(*) FILTER (WHERE loyalty_card = true)
        * 100.0 / COUNT(*), 1
    )                                                AS pct_con_lealtad

FROM transactions
""")

📍 **Qué observar:** Los números de `sin_customer_id` y `loyalty_false` deben ser iguales.
Si coinciden → tiene sentido: sin tarjeta = anónimo.

✅ **Decisión:** Aceptar. Para cohortes usar solo `loyalty_card = TRUE`.

---
## 2️⃣ CONSISTENCIA — ¿Los números cuadran?
> **Pregunta:** ¿El `total_amount` coincide con `SUM(unit_price * quantity)` de los items?

In [ ]:
sql("""
WITH suma_items AS (
    -- Paso 1: sumar lineas por transaccion
    SELECT
        transaction_id,
        SUM(unit_price * quantity) AS suma_calculada
    FROM tx_items
    GROUP BY transaction_id
),
comparacion AS (
    -- Paso 2: juntar con el total oficial
    SELECT
        t.transaction_id,
        t.total_amount,
        s.suma_calculada,
        ABS(t.total_amount - s.suma_calculada) AS diferencia
    FROM transactions t
    JOIN suma_items s USING (transaction_id)
)
-- Paso 3: contar discrepancias
SELECT
    COUNT(*)                                              AS transacciones_comparadas,
    COUNT(*) FILTER (WHERE diferencia > 0.02)             AS con_discrepancia,
    ROUND(
        COUNT(*) FILTER (WHERE diferencia > 0.02)
        * 100.0 / COUNT(*), 2
    )                                                     AS pct_discrepancia,
    ROUND(MAX(diferencia), 2)                             AS diferencia_maxima,
    ROUND(AVG(diferencia) FILTER (WHERE diferencia > 0.02), 2) AS diferencia_promedio
FROM comparacion
""")

In [ ]:
# Ver las 5 con mayor discrepancia
sql("""
WITH suma_items AS (
    SELECT transaction_id, SUM(unit_price * quantity) AS suma_calculada
    FROM tx_items GROUP BY transaction_id
)
SELECT
    t.transaction_id,
    t.store_id,
    t.total_amount,
    ROUND(s.suma_calculada, 2) AS suma_items,
    ROUND(ABS(t.total_amount - s.suma_calculada), 2) AS diferencia
FROM transactions t
JOIN suma_items s USING (transaction_id)
WHERE ABS(t.total_amount - s.suma_calculada) > 0.02
ORDER BY diferencia DESC
LIMIT 5
""")

✅ **Decisión:** 1% de discrepancia es tolerable (descuentos/redondeos). Usar `total_amount` como GMV oficial.

---
## 3️⃣ UNICIDAD — ¿Hay duplicados?
> **Pregunta:** ¿Existen `transaction_id` repetidos en la tabla de transacciones?

In [ ]:
sql("""
SELECT
    'transactions'       AS tabla,
    COUNT(*)             AS total_filas,
    COUNT(DISTINCT transaction_id)     AS ids_unicos,
    COUNT(*) - COUNT(DISTINCT transaction_id) AS duplicados
FROM transactions

UNION ALL

SELECT
    'tx_items',
    COUNT(*),
    COUNT(DISTINCT transaction_item_id),
    COUNT(*) - COUNT(DISTINCT transaction_item_id)
FROM tx_items
""")

In [ ]:
# Si hay duplicados, ver cuáles son:
sql("""
SELECT transaction_id, COUNT(*) AS veces
FROM transactions
GROUP BY transaction_id
HAVING COUNT(*) > 1
ORDER BY veces DESC
LIMIT 10
""")

✅ **Decisión:** 0 duplicados. Sin acción requerida. 🎉

---
## 4️⃣ VALIDEZ — ¿Los valores tienen sentido?
> **Preguntas:** ¿Hay `total_amount` negativos/cero? ¿Hay `unit_price=0` sin promo?

In [ ]:
# total_amount negativos o cero
sql("""
SELECT
    COUNT(*) FILTER (WHERE total_amount <= 0)   AS montos_invalidos,
    COUNT(*) FILTER (WHERE total_amount < 0)    AS montos_negativos,
    COUNT(*) FILTER (WHERE total_amount = 0)    AS montos_cero,
    MIN(total_amount)                           AS minimo,
    MAX(total_amount)                           AS maximo
FROM transactions
""")

In [ ]:
# Ver las filas con monto invalido
sql("""
SELECT transaction_id, store_id, transaction_date,
       total_amount, status, payment_method
FROM transactions
WHERE total_amount <= 0
""")

In [ ]:
# unit_price = 0 en items
sql("""
SELECT
    COUNT(*) FILTER (WHERE unit_price = 0)                        AS precio_cero_total,
    COUNT(*) FILTER (WHERE unit_price = 0 AND was_on_promo = false) AS precio_cero_sin_promo,
    COUNT(*) FILTER (WHERE unit_price = 0 AND was_on_promo = true)  AS precio_cero_con_promo,
    ROUND(
        COUNT(*) FILTER (WHERE unit_price = 0) * 100.0 / COUNT(*), 2
    )                                                             AS pct_precio_cero
FROM tx_items
""")

✅ **Decisión:** Excluir `total_amount <= 0` del GMV. Excluir `unit_price=0` sin promo del GMROI.

---
## 5️⃣ INTEGRIDAD REFERENCIAL — ¿Todo está conectado?
> **Pregunta:** ¿Hay `store_id` en transactions que no existan en stores?
> ¿Hay `vendor_id` en products que no existan en vendors?

In [ ]:
# store_id huerfanos en transactions
sql("""
SELECT
    t.store_id,
    COUNT(*) AS transacciones_huerfanas
FROM transactions t
LEFT JOIN stores s USING (store_id)
WHERE s.store_id IS NULL        -- no existe en stores
GROUP BY t.store_id
ORDER BY transacciones_huerfanas DESC
""")

In [ ]:
# vendor_id huerfanos en products
sql("""
SELECT
    p.vendor_id,
    COUNT(*) AS productos_afectados
FROM products p
LEFT JOIN vendors v USING (vendor_id)
WHERE v.vendor_id IS NULL       -- no existe en vendors
GROUP BY p.vendor_id
""")

✅ **Decisión:** 0 tiendas huérfanas. VND_031 no existe en vendors (5 productos afectados) → excluir del GMROI.

---
## 6️⃣ FRESCURA — ¿Hay gaps de días sin ventas?
> **Pregunta:** ¿Hay tiendas con 3+ días consecutivos sin transacciones?
> Usamos `LAG()` — la misma técnica del Bloque 1 Query 5!

In [ ]:
sql("""
WITH dias_venta AS (
    -- Dias unicos de venta por tienda
    SELECT DISTINCT
        store_id,
        CAST(transaction_date AS DATE) AS fecha_venta
    FROM transactions
),
con_lag AS (
    -- Calcular cuantos dias pasaron desde la venta anterior
    SELECT
        store_id,
        fecha_venta,
        LAG(fecha_venta) OVER (
            PARTITION BY store_id
            ORDER BY fecha_venta
        ) AS venta_anterior,
        DATEDIFF('day',
            LAG(fecha_venta) OVER (
                PARTITION BY store_id ORDER BY fecha_venta
            ),
            fecha_venta
        ) - 1 AS dias_sin_ventas
    FROM dias_venta
)
-- Solo gaps de 3+ dias (sospechosos)
SELECT
    store_id,
    venta_anterior  AS gap_inicio,
    fecha_venta     AS gap_fin,
    dias_sin_ventas,
    CASE
        WHEN dias_sin_ventas BETWEEN 3 AND 7  THEN 'REVISAR'
        WHEN dias_sin_ventas BETWEEN 8 AND 14 THEN 'ALERTA'
        WHEN dias_sin_ventas > 14             THEN 'CRITICO'
    END AS severidad
FROM con_lag
WHERE dias_sin_ventas >= 3
ORDER BY dias_sin_ventas DESC
""")

✅ **Decisión:** 1 gap de 7 días (REVISAR). Puede ser inventario o festivo. Gaps > 14 días se excluirían de Comp Sales.

---
## 7️⃣ INTEGRIDAD TEMPORAL — ¿Ventas antes de que abriera la tienda?
> **Pregunta:** ¿Existe alguna tienda con transacciones anteriores a su `opening_date`?

In [ ]:
sql("""
SELECT
    t.store_id,
    s.opening_date,
    MIN(t.transaction_date)  AS primera_tx,
    COUNT(*)                 AS tx_antes_apertura
FROM transactions t
JOIN stores s USING (store_id)
WHERE CAST(t.transaction_date AS DATE) < CAST(s.opening_date AS DATE)
GROUP BY t.store_id, s.opening_date
ORDER BY tx_antes_apertura DESC
""")

✅ **Decisión:** 50 transacciones en 1 tienda antes de su apertura. Son errores de carga → EXCLUIR.

---
## 8️⃣ INTEGRIDAD DEL A/B TEST — ¿Grupos contaminados?
> **Pregunta:** ¿Hay tiendas asignadas a CONTROL y TREATMENT simultáneamente?

In [ ]:
sql("""
WITH variantes AS (
    -- Que variantes tiene cada tienda por promocion
    SELECT
        store_id,
        promo_name,
        COUNT(DISTINCT variant) AS num_variantes,
        STRING_AGG(DISTINCT variant, ' + ') AS grupos_asignados
    FROM promotions
    GROUP BY store_id, promo_name
)
-- Filtrar las que tienen ambos grupos
SELECT
    store_id,
    promo_name,
    grupos_asignados,
    'EXCLUIR DEL AB TEST' AS accion
FROM variantes
WHERE num_variantes > 1
""")

In [ ]:
# Resumen del experimento: cuantas tiendas en cada grupo
sql("""
SELECT
    variant,
    COUNT(DISTINCT store_id) AS tiendas
FROM promotions
WHERE store_id NOT IN ('TIENDA_008', 'TIENDA_037')  -- excluir contaminadas
GROUP BY variant
""")

✅ **Decisión:** TIENDA_008 y TIENDA_037 en ambos grupos → EXCLUIR del A/B test.

---
## 🌟 BONUS: Reporte Final en una sola query
> ¡Una query que resume TODA la auditoría de golpe!

In [ ]:
sql("""
SELECT
    -- Completitud
    ROUND(COUNT(*) FILTER (WHERE customer_id IS NULL) * 100.0 / COUNT(*), 1)
        AS pct_sin_customer_id,
    ROUND(COUNT(*) FILTER (WHERE loyalty_card = true) * 100.0 / COUNT(*), 1)
        AS pct_con_lealtad,

    -- Validez
    COUNT(*) FILTER (WHERE total_amount <= 0)
        AS montos_invalidos,
    COUNT(*) FILTER (WHERE status != 'COMPLETED')
        AS transacciones_no_completadas,

    -- Unicidad
    COUNT(*) - COUNT(DISTINCT transaction_id)
        AS transaction_id_duplicados,

    -- Rango de fechas
    MIN(CAST(transaction_date AS DATE)) AS fecha_minima,
    MAX(CAST(transaction_date AS DATE)) AS fecha_maxima,
    COUNT(DISTINCT store_id)            AS tiendas_activas

FROM transactions
""")

---
## 📝 Resumen de Decisiones

| Dimensión | Hallazgo | Decisión |
|---|---|---|
| Completitud | 59.8% sin customer_id | Aceptar. Usar loyalty=TRUE para cohortes |
| Consistencia | 1% discrepancia total_amount | Usar `total_amount` como fuente de verdad |
| Unicidad | 0 duplicados | Sin acción |
| Validez | 3 montos ≤0, 231 precios $0 | Excluir de GMV y GMROI |
| Integridad Ref. | VND_031 huérfano (5 productos) | Excluir del GMROI |
| Frescura | 1 gap de 7 días | Monitorear, aceptar si < 14 días |
| Integridad Temporal | 50 tx antes de apertura en 1 tienda | Excluir |
| A/B Test | TIENDA_008 y 037 en ambos grupos | Excluir del experimento |

---
*Auditoría SQL con DuckDB — compatible con BigQuery cambiando solo los nombres de tabla*